# Data Reading & File Formats

This module covers reading structured data from Unity Catalog Volumes using Spark DataFrameReader (CSV, JSON, Parquet), the SQL-native `read_files()` function, CTAS for one-shot table creation, and Lakeflow Connect for zero-code SaaS ingestion.

> For **incremental ingestion** (COPY INTO, Auto Loader, Structured Streaming) — see **M05**.

## Learning Objectives

After completing this module you will be able to:

- **Read** CSV, JSON, and Parquet files from Unity Catalog Volumes using `spark.read`
- **Define** explicit `StructType` schemas for type safety and performance
- **Explain** the performance difference between `inferSchema` and manual schema
- **Use** `read_files()` as the SQL-native Volume reader (DBR 13.3+)
- **Apply** CTAS to create a Delta table from a SELECT in one statement
- **Describe** Lakeflow Connect as a zero-code SaaS ingestion option
- **Explore** a loaded DataFrame using `display()`, `describe()`, `summary()`, and `count()`

| Exam Domain | Weight |
|---|---|
| Development & Ingestion | 30% |



## Setup

Initialize the environment, import libraries, and prepare simulated data sources for all ingestion demos in this module.

In [0]:
%run ../../setup/00_setup

### Configuration

Import libraries and define paths for source data, checkpoints, and schema locations.

In [0]:

# PySpark types for schema definition
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, DateType, ArrayType
)

# PySpark functions for transformations and aggregations
from pyspark.sql.functions import (
    col, lit, concat, upper, lower, year, month, day,
    sum, avg, min, max, count, desc, asc,
    to_date, current_date, datediff, when
)

# Keep the namespace import as well for convenience
from pyspark.sql import functions as F

from datetime import datetime
import time


In [0]:
print(BRONZE_SCHEMA)

In [0]:
# Set default catalog and schema
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

In [0]:
print(DATASET_PATH)

In [0]:
# Paths to data directories (subdirectories in DATASET_PATH from 00_setup)
CUSTOMERS_PATH = f"{DATASET_PATH}/customers"
ORDERS_PATH = f"{DATASET_PATH}/orders"
PRODUCTS_PATH = f"{DATASET_PATH}/products"

# Paths to specific files
CUSTOMERS_CSV = f"{CUSTOMERS_PATH}/customers.csv"
ORDERS_JSON = f"{ORDERS_PATH}/orders_batch.json"
PRODUCTS_PARQUET = f"{PRODUCTS_PATH}/products.parquet"

In [0]:
display(
    spark.createDataFrame([
        ("CATALOG", CATALOG),
        ("BRONZE_SCHEMA", BRONZE_SCHEMA),
        ("CUSTOMERS_CSV", CUSTOMERS_CSV),
        ("ORDERS_JSON", ORDERS_JSON),
        ("PRODUCTS_PARQUET", PRODUCTS_PARQUET),
    ], ["Variable", "Value"])
)

### Ingestion Methods — Module Overview

<img src="../../../assets/images/207e295b292949f48861279ed69da983.webp" width="800">

| Feature | CTAS | COPY INTO | Auto Loader |
|---------|------|-----------|-------------|
| **Incremental** | No | Yes | Yes |
| **Idempotent** | No | Yes | Yes |
| **Schema Evolution** | No | Limited | Advanced |
| **File Tracking** | No | Metadata | Checkpoint |
| **Scalability** | Low | Medium | High |
| **Streaming** | No | No | Yes |
| **Use Case** | One-time | Scheduled batch | Real-time/Streaming |

> **Pro Tip:** Auto Loader (`cloudFiles`) is the **recommended** ingestion method for new projects. COPY INTO is suitable for scheduled batch loads with <100K files.

> **COPY INTO** and **Auto Loader** are covered in **M05: Incremental Ingestion**.

### Data Processing Patterns

![Data Processing Patterns — Full Batch, Incremental, Continuous, Trigger Once](../../../assets/images/1ba507cd604849878e74e586f4df3559.png)


## Spark DataFrameReader — File Formats

## CSV Import (Customers)

> **Definition (Databricks):** `spark.read.format("csv")` — a `DataFrameReader` method that reads delimited text files. `header=true` treats the first row as column names; `inferSchema=true` causes Spark to read the file twice — once to detect types, once to load data (expensive on large datasets). The recommended production approach is to provide an explicit schema (`StructType`), which eliminates the extra scan and guarantees data types.

Loading CSV files into DataFrames using `spark.read` with schema inference and manual schema definition. We'll work with customer data throughout this section.

### Auto Schema Inference

Using `inferSchema=true` to let Spark detect column types automatically.


In [0]:
print(CUSTOMERS_CSV)

In [0]:
# Load CSV data with automatic schema inference
customers_auto_df = (
    spark.read
    .format("csv")
    .option("header", "true")       # First line contains column names
    .option("inferSchema", "true")  # Automatic data type inference
    .load(CUSTOMERS_CSV)
)

In [0]:
# Verify loaded data
customers_auto_df.printSchema()
# display() — Databricks-specific function: renders an interactive table
# Pipeline stage: BRONZE — preview raw AutoLoader ingest (triggers Spark execution)


In [0]:
# Always pair with .limit(n) to avoid full table scan
display(customers_auto_df)

## DataFrame Exploration

After loading data, use these functions to explore a DataFrame interactively:

| Method | Returns | Purpose |
|--------|---------|----------|
| `display(df)` | Interactive table | Databricks-native rich UI, supports charts, sorting, search |
| `df.show(n)` | Text output | Standard Spark, works anywhere (JVM, local) |
| `df.printSchema()` | Schema tree | Column names, types, nullability |
| `df.count()` | Long | Total row count (triggers full scan) |
| `df.describe()` | DataFrame | Basic stats: count, mean, stddev, min, max (numeric + string cols) |
| `df.summary()` | DataFrame | Extended stats: adds percentiles (25%, 50%, 75%) |

> **Exam tip:** `describe()` and `summary()` both return a DataFrame — you can further filter them with `.filter()` or `.where()`.

In [0]:
# -- DataFrame Exploration --
# display() — Databricks-native rich table with interactive chart support
display(customers_auto_df)

# show(n) — Standard Spark text output
customers_auto_df.show(3, truncate=False)

# describe() — basic statistics: count, mean, stddev, min, max
display(customers_auto_df.describe())

# summary() — extended statistics: adds 25%, 50%, 75% percentiles
display(customers_auto_df.summary())

# count() — total row count (triggers full Spark scan)
print(f"Row count: {customers_auto_df.count()}")

### Extended Reader Options

All important CSV options for production use — go beyond the basics:

| Option | Type | Default | Description |
|---|---|---|---|
| `header` | bool | `false` | First row as column names |
| `inferSchema` | bool | `false` | Auto-detect types (2 scans — expensive!) |
| `delimiter` / `sep` | string | `,` | Column separator (e.g. `;`, `\t`) |
| `quote` | string | `"` | Quote character for text fields |
| `escape` | string | `\` | Escape character inside quoted strings |
| `nullValue` | string | `""` | String to treat as `null` (e.g. `"N/A"`) |
| `emptyValue` | string | `""` | Value to replace empty strings |
| `nanValue` | string | `NaN` | String to treat as `NaN` |
| `dateFormat` | string | `yyyy-MM-dd` | Pattern for `DateType` columns |
| `timestampFormat` | string | ISO 8601 | Pattern for `TimestampType` columns |
| `encoding` | string | `UTF-8` | File encoding (e.g. `ISO-8859-1`, `UTF-16`) |
| `multiLine` | bool | `false` | Allow newlines inside quoted fields |
| `comment` | string | disabled | Lines starting with this char are skipped |
| `ignoreLeadingWhiteSpace` | bool | `false` | Strip leading whitespace |
| `ignoreTrailingWhiteSpace` | bool | `false` | Strip trailing whitespace |
| `mode` | string | `PERMISSIVE` | Error handling: `PERMISSIVE` / `DROPMALFORMED` / `FAILFAST` |
| `columnNameOfCorruptRecord` | string | `_corrupt_record` | Column to store malformed rows (PERMISSIVE mode) |
| `maxColumns` | int | `20480` | Max number of columns allowed |
| `samplingRatio` | double | `1.0` | Fraction of rows to scan for `inferSchema` |

```python
# Production-grade CSV read with most common options
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")                          # semicolon-delimited
    .option("encoding", "UTF-8")
    .option("nullValue", "N/A")
    .option("emptyValue", "UNKNOWN")
    .option("dateFormat", "dd/MM/yyyy")
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .option("mode", "DROPMALFORMED")             # silently drop bad rows
    .option("multiLine", "true")                 # allow newlines in fields
    .schema(my_schema)
    .load("/Volumes/catalog/schema/volume/data.csv")
)
```

> ** Tip: ** `PERMISSIVE` (default) stores bad rows in `_corrupt_record`. `FAILFAST` raises an exception on first bad row. Use `DROPMALFORMED` to silently skip them.


### Manual Schema Definition

Defining an explicit `StructType` schema for type safety and performance.

In [0]:
customers_auto_df.printSchema()

In [0]:
# Schema definition for customers
# Structure: customer_id (string), first_name (string), last_name (string), email (string), city (string), country (string), registration_date (timestamp)
customers_schema = StructType([
    StructField("customer_id", StringType(), nullable=False),
    StructField("first_name", StringType(), nullable=True),
    StructField("last_name", StringType(), nullable=True),
    StructField("city", StringType(), nullable=True),
    StructField("email", StringType(), nullable=True),
    StructField("country", StringType(), nullable=True),
    StructField("registration_date", TimestampType(), nullable=True)
])

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), nullable=False),
    StructField("first_name", StringType(), nullable=True),
    StructField("last_name", StringType(), nullable=True),
    StructField("email", StringType(), nullable=True),
    StructField("phone", StringType(), nullable=True),
    StructField("city", StringType(), nullable=True),
    StructField("state", StringType(), nullable=True),
    StructField("country", StringType(), nullable=True),
    StructField("registration_date", StringType(), nullable=True)
])

In [0]:
# Load CSV data with manually defined schema
customers_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customers_schema)
    .load(CUSTOMERS_CSV)
)

In [0]:
# Verify loaded data
customers_df.printSchema()
display(customers_df.limit(5))

In [0]:
customers_df = customers_auto_df

## JSON Import (Orders)

> **Definition (Databricks):** `spark.read.format("json")` — a `DataFrameReader` for JSON files in *JSON Lines* format (one JSON object per line). Spark automatically handles primitive types, arrays and nested structures (`StructType`). For large datasets, providing an explicit schema is recommended instead of `inferSchema`, as Spark must scan the entire file to detect types — especially costly with deeply nested structures.

Reading JSON files into DataFrames with automatic and manual schema approaches. JSON supports nested structures natively.

### Auto Schema Inference

Letting Spark infer the JSON schema automatically.


In [0]:
# Load JSON data with automatic schema inference
orders_auto_df = (
    spark.read
    .format("json")
    .option("inferSchema", "true")
    .load(ORDERS_JSON)
)

In [0]:
# Verify loaded data
orders_auto_df.printSchema()
display(orders_auto_df.limit(5))

### JSON Reader Options

Providing an explicit schema to select and type only the columns we need.

| Option | Type | Default | Description |
|---|---|---|---|
| `inferSchema` | bool | `false` | Auto-detect types (full file scan!) |
| `multiLine` | bool | `false` | Allow JSON objects spanning multiple lines |
| `allowComments` | bool | `false` | Allow `//` and `/* */` comments in JSON |
| `allowUnquotedFieldNames` | bool | `false` | Allow keys without quotes |
| `allowSingleQuotes` | bool | `false` | Allow single-quoted strings |
| `allowNumericLeadingZeros` | bool | `false` | Allow `007` as integer |
| `allowNonNumericNumbers` | bool | `false` | Allow `NaN`, `INF`, `-INF` |
| `dateFormat` | string | `yyyy-MM-dd` | Pattern for date parsing |
| `timestampFormat` | string | ISO 8601 | Pattern for timestamp parsing |
| `columnNameOfCorruptRecord` | string | `_corrupt_record` | Store unparseable rows |
| `mode` | string | `PERMISSIVE` | `PERMISSIVE` / `DROPMALFORMED` / `FAILFAST` |
| `primitivesAsString` | bool | `false` | Read all primitives as strings |
| `dropFieldIfAllNull` | bool | `false` | Drop columns that are all null |
| `encoding` | string | `UTF-8` | File encoding |
| `samplingRatio` | double | `1.0` | Fraction of rows to scan for `inferSchema` |

```python
# Multi-line JSON (one object across multiple lines)
df = (
    spark.read
    .format("json")
    .option("multiLine", "true")
    .option("mode", "DROPMALFORMED")
    .schema(my_schema)
    .load("/Volumes/catalog/schema/volume/data.json")
)
```


In [0]:
# Schema definition for orders
# Actual structure: order_id, customer_id, product_id, store_id, order_datetime, quantity, unit_price, discount_percent, total_amount, payment_method
orders_schema = StructType([
    StructField("order_id", StringType(), nullable=True),  # Can be null in data
    StructField("customer_id", StringType(), nullable=False),
    StructField("order_datetime", StringType(), nullable=True),  # String, will convert to timestamp later
    StructField("total_amount", DoubleType(), nullable=False),
    StructField("payment_method", StringType(), nullable=False)
])

%sql


CREATE TABLE 
(
    order_id nvarchar(max),customer_id nvarchar(max)....,total_amount int)
)

In [0]:
# Load JSON data with manually defined schema
orders_df = (
    spark.read
    .format("json")
    .schema(orders_schema)
    .load(ORDERS_JSON)
)

In [0]:
# Verify loaded data
orders_df.printSchema()
display(orders_df.limit(5))

## Parquet Import (Products)

> **Definition (Databricks):** `spark.read.format("parquet")` — a `DataFrameReader` for Parquet files — the columnar binary format that is the default in Databricks and Delta Lake. Parquet stores the schema inside the file (no `inferSchema` or manual schema needed), supports *column pruning* and *predicate pushdown*, which significantly reduces the amount of data read. It is the recommended format for all Lakehouse layers (Bronze → Gold).

Reading Parquet files — the preferred columnar format in Databricks. Parquet has built-in schema, so no manual definition is needed.



In [0]:
# Parquet already contains built-in schema - no need to define it
products_df = (
    spark.read
    .format("parquet")
    .load(PRODUCTS_PARQUET)
)

In [0]:
# Verify loaded data
products_df.printSchema()
display(products_df.limit(5))

## Performance: inferSchema vs Manual Schema

> **Definition (Databricks):** `inferSchema=true` causes Spark to perform **two full scans** of the data source: the first to infer column types, the second to load data. With an explicit `StructType`, Spark performs **one scan** according to the provided schema. Databricks recommends explicit schemas in production environments — especially when reading from cloud object stores (S3, ADLS), where every scan generates additional I/O operation costs and execution time.

Comparing read performance between automatic schema inference and explicit schema definition to demonstrate why manual schemas are recommended in production.



In [0]:
# Automatic schema inference
start_auto = time.time()
df_auto = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CUSTOMERS_CSV)
)
count_auto = df_auto.count()  # Action - forces execution
time_auto = time.time() - start_auto

In [0]:
# Manual schema definition
start_manual = time.time()
df_manual = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customers_schema)
    .load(CUSTOMERS_CSV)
)
count_manual = df_manual.count()  # Action - forces execution
time_manual = time.time() - start_manual

In [0]:
display(df_manual)

In [0]:
# Comparison
speedup = (time_auto - time_manual) / time_auto * 100
print(f"\nConclusion: Manual schema is faster by {speedup:.1f}%")

## read_files() — Unity Catalog Native Reader

> **Definition (Databricks):** `read_files(path, format, ...)` — a native SQL function in Databricks Runtime 13.3+ that enables reading files directly in SQL queries without creating an external table. Works with Unity Catalog **Volumes** (paths `/Volumes/catalog/schema/volume/file`). Automatically attaches file metadata (`_metadata.file_path`, `_metadata.file_name`, `_metadata.file_size`). Supported formats: `csv`, `json`, `parquet`, `avro`, `orc`, `text`, `binaryFile`. Recommended by Databricks for SQL-first workflows with UC — replaces `CREATE TABLE USING` with an external location.

| Feature | `read_files()` | `spark.read` |
|---|---|---|
| Language | SQL | Python |
| Schema inference | Automatic | Manual or `inferSchema` |
| UC integration | Native | Via path |
| Use case | SQL-first workflows | Programmatic pipelines |

```sql
-- Read CSV from a Volume
SELECT * FROM read_files('/Volumes/catalog/schema/volume/file.csv', format => 'csv', header => true);

-- Create table from files
CREATE TABLE bronze.customers AS SELECT * FROM read_files('/Volumes/.../file.csv');
```

> **Exam Tip:** `read_files()` is the recommended way to read files in SQL workflows with Unity Catalog.


In [0]:
%sql

SELECT *, _metadata.file_path
  FROM read_files(
    '/Volumes/retailhub_trainer/default/datasets/customers/customers.csv',
    format => 'csv',
    header => true)


In [0]:
# Read CSV from a Unity Catalog Volume using read_files() — SQL-native reader (DBR 13.3+)
# The path follows the UC Volume convention: /Volumes/<catalog>/<schema>/<volume>/file
display(spark.sql(f"""
  SELECT *, _metadata.file_path
  FROM read_files(
    '/Volumes/{CATALOG}/default/datasets/customers/customers.csv',
    format => 'csv',
    header => true
  )
"""))

In [0]:
# CTAS using read_files() — creates a managed Delta table from files in a Volume
spark.sql(f"""
  CREATE OR REPLACE TABLE {BRONZE_SCHEMA}.customers_rf
  AS SELECT *, _metadata.file_path
  FROM read_files(
    '/Volumes/{CATALOG}/default/datasets/customers/customers.csv',
    format => 'csv',
    header => true
  )
  WHERE customer_segment = 'Premium'
""")


display(spark.table(f"{BRONZE_SCHEMA}.customers_rf"))

In [0]:
%sql 
SELECT * FROM bronze.customers_rf

### CTAS — Create Table As Select

The simplest way to create a Delta table from a query result. CTAS reads data once and writes a snapshot — no tracking, no incremental processing.

```sql
CREATE OR REPLACE TABLE target AS
SELECT ... FROM source
```

| Feature | CTAS |
|---------|------|
| **Incremental** | No — full overwrite every time |
| **Idempotent** | No — `CREATE OR REPLACE` drops & recreates |
| **Schema evolution** | No — fixed at creation |
| **File tracking** | No |
| **Best for** | One-time loads, prototyping, small tables |

> **Key limitation:** CTAS does NOT track which files were already loaded. Every re-run overwrites the entire table. For incremental loading, use **COPY INTO** or **Auto Loader**.

In [0]:
# Example: Load CSV from volume to customer_cts table using CTAS

table_name = "customer_cts"

display(spark.sql(f"""
CREATE OR REPLACE TABLE {table_name} AS
SELECT *
FROM csv.`{CUSTOMERS_CSV}`
"""))

In [0]:
%sql
select * from customer_cts

**CTAS with transformations — type casting, filtering, computed columns:**

In [0]:
# CTAS with transformations — real-world pattern

display(spark.sql(f"""
CREATE OR REPLACE TABLE customer_cts_enriched AS
SELECT
  customer_id,
  UPPER(first_name) AS first_name,
  UPPER(last_name) AS last_name,
  LOWER(email) AS email,
  city,
  state,
  country,
  TO_DATE(registration_date, 'yyyy-MM-dd') AS registration_date,
  customer_segment,
  DATEDIFF(current_date(), TO_DATE(registration_date, 'yyyy-MM-dd')) AS days_since_registration,
  current_timestamp() AS _ingestion_timestamp
FROM read_files('{CUSTOMERS_CSV}', format => 'csv', header => true)
WHERE customer_segment IS NOT NULL
"""))

display(spark.table("customer_cts_enriched"))

**CTAS is NOT incremental — re-run overwrites everything:**

In [0]:
# [1/2] Record row count before re-run
count_before = spark.table("customer_cts").count()
print(f"Rows before re-run: {count_before}")

In [0]:
# [2/2] Re-run CTAS on the same source — count stays the same (full overwrite, not append)
spark.sql(f"""
CREATE OR REPLACE TABLE customer_cts AS
SELECT * FROM csv.`{CUSTOMERS_CSV}`
""")

count_after = spark.table("customer_cts").count()

display(spark.createDataFrame([
    ("Before CTAS re-run", count_before),
    ("After CTAS re-run",  count_after),
    ("Difference",         count_after - count_before),
    ("Conclusion",         0),  # same count — full overwrite, not append
], ["State", "Count"]))

# Unlike COPY INTO or Auto Loader, CTAS doesn't track files.
# It always processes ALL source files from scratch.
print("CTAS = full overwrite. No file tracking. No incremental.")

**CTAS from different formats — Parquet, JSON, Delta:**

```sql
-- From Parquet
CREATE TABLE bronze_parquet AS SELECT * FROM parquet.`/path/to/files/`;

-- From JSON
CREATE TABLE bronze_json AS SELECT * FROM json.`/path/to/files/`;

-- From existing Delta table (snapshot copy)
CREATE TABLE gold_snapshot AS SELECT * FROM silver_table WHERE date = current_date();

-- Read from Volume path
CREATE TABLE bronze_volume AS SELECT * FROM read_files('/Volumes/catalog/schema/volume/data/');
```

> **Pro Tip:** Use `read_files()` for Volume-based data — it's the recommended approach in Unity Catalog environments. Direct format readers (`csv.`, `json.`) work but `read_files()` adds metadata columns automatically.

## Lakeflow Connect — Managed SaaS Ingestion

**Lakeflow Connect** is a built-in, no-code Databricks service for ingesting data from SaaS applications directly into the Lakehouse.

Unlike Auto Loader (code-based, files on cloud storage), Lakeflow Connect uses **managed connectors** that handle authentication, pagination, and incremental extraction automatically.

### How it works
1. Configure a connector in **Data Engineering → Lakeflow Connect** UI
2. Authenticate with the SaaS application (OAuth, API key, etc.)
3. Select objects/tables to ingest
4. Databricks creates a **Lakeflow Pipeline** that loads data incrementally
5. Data lands in Unity Catalog Bronze tables — ready for downstream transformation

> **No code required** — the entire pipeline is managed by Databricks.


### Connector Categories

| Category | Examples | Notes |
|----------|----------|-------|
| **CRM** | Salesforce, HubSpot | Full + incremental load |
| **ERP** | SAP, Oracle ERP | Batch scheduled |
| **Finance & HR** | Workday, Netsuite | Pre-built API handling |
| **Databases** | MySQL, PostgreSQL, SQL Server | CDC via log reader |
| **Partner connectors** | Fivetran, Airbyte, Stitch | Partner ecosystem |

### Comparison: Ingestion methods

| Method | Source | Code | Schedule | Use Case |
|--------|--------|------|----------|----------|
| **COPY INTO** | Cloud storage (S3/ADLS/GCS) | SQL | Manual / Jobs | Batch files, idempotent |
| **Auto Loader** | Cloud storage | Python | Streaming / Jobs | High-volume, schema evolution |
| **read_files()** | Volumes (UC) | SQL | Ad-hoc / CTAS | SQL-first, UC native |
| **Lakeflow Connect** | SaaS apps, databases | None (UI) | Managed pipeline | CRM/ERP integration |
| **JDBC/ODBC** | Relational databases | Python/SQL | Manual / Jobs | Custom DB connectivity |

> **Exam Tip:** Lakeflow Connect = zero-code managed connectors. Auto Loader = code-based streaming from files.


## JDBC/ODBC — Relational Database Connectivity

For databases that are not supported by Lakeflow Connect, use **JDBC** (Java Database Connectivity) to read data directly.

```python
# Read from PostgreSQL via JDBC
df = (spark.read
    .format("jdbc")
    .option("url", "jdbc:postgresql://host:5432/dbname")
    .option("dbtable", "schema.orders")
    .option("user", dbutils.secrets.get("scope", "db_user"))
    .option("password", dbutils.secrets.get("scope", "db_pass"))
    .option("numPartitions", 8)
    .option("partitionColumn", "order_id")
    .option("lowerBound", 1)
    .option("upperBound", 1000000)
    .load())
```

### Key Options

| Option | Purpose | Example |
|--------|---------|--------|
| `numPartitions` | Parallel reads | `8` |
| `partitionColumn` | Column for splitting | `order_id` |
| `fetchsize` | Rows per fetch | `10000` |
| `pushDownPredicate` | Filter at source | `true` (default) |

> **Security:** Always use `dbutils.secrets.get()` for credentials — never hardcode passwords in notebooks.

> **Exam Tip:** JDBC credentials must be stored in Databricks Secrets. Use `dbutils.secrets.get(scope, key)`.


## Summary

| Topic | Key Concept | Exam Keywords |
|---|---|---|
| **CSV** | `header`, `inferSchema`, explicit schema | `StructType`, `StructField`, `.schema()` |
| **JSON** | JSON Lines format, nested structs | `multiLine`, `inferSchema` |
| **Parquet** | Schema embedded, columnar, predicate pushdown | `format("parquet")` |
| **Schema Performance** | Manual = 1 scan vs inferSchema = 2 scans | Production best practice |
| **read_files()** | SQL-native Unity Catalog Volume reader (DBR 13.3+) | `read_files()` |
| **CTAS** | Create Delta table from SELECT — not incremental | `CREATE TABLE AS SELECT` |
| **Lakeflow Connect** | Zero-code SaaS ingestion (UI only) | Salesforce, Workday, SAP |

> For incremental loading (COPY INTO, Auto Loader, Streaming) — **→ M05: Incremental Ingestion**



## DataFrame Transformations *(optional — if time permits)*

Core PySpark transformations used throughout the pipeline: selecting columns, adding/modifying columns, type casting, filtering, conditional logic, and basic aggregations.

| Transformation | Method | SQL equivalent |
|---|---|---|
| Column projection | `select()` | `SELECT col1, col2` |
| Add / modify column | `withColumn()` | computed column |
| Type conversion | `cast()` | `CAST(col AS type)` |
| Filter rows | `filter()` / `where()` | `WHERE` |
| Conditional values | `when().otherwise()` | `CASE WHEN` |
| Aggregation | `groupBy().agg()` | `GROUP BY` + aggregate functions |


### select — Column Projection

> **Definition (Databricks):** `DataFrame.select(*cols)` — returns a new DataFrame containing only the specified columns or expressions. Equivalent to `SELECT` in SQL. Arguments can be column names (string), `Column` objects (`col()`), or expressions. Every PySpark transformation is immutable — `select()` does not modify the original DataFrame.


In [0]:
# Select specific columns
customers_selected = customers_df.select("customer_id", "first_name", "last_name", "email")
display(customers_selected.limit(5))

# Select with expressions
customers_transformed = customers_df.select(
    col("customer_id"),
    upper(col("first_name")).alias("first_name_upper"),
    col("email")
)
display(customers_transformed.limit(5))


### withColumn — Add / Modify Columns

> **Definition (Databricks):** `DataFrame.withColumn(colName, col)` — returns a new DataFrame by adding a new column or replacing an existing one. The method is **lazy** — no computation happens until an action (`count()`, `show()`, `write`) is called. Chain multiple `withColumn()` calls freely; for many columns at once, using `select()` with expressions is more efficient.


In [0]:
# Add computed columns
customers_enriched = (customers_df
    .withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name")))
    .withColumn("email_lower", lower(col("email")))
    .withColumn("registration_year", year(col("registration_date")))
)
display(customers_enriched.select(
    "customer_id", "full_name", "email_lower", "registration_year"
).limit(5))


### cast — Type Conversion

> **Definition (Databricks):** `Column.cast(dataType)` — converts the column to the specified type. If conversion fails (e.g. `"abc"` → `IntegerType`), the result is `null` (PERMISSIVE mode). The type can be given as a `DataType` object or a SQL string (`"int"`, `"date"`, `"timestamp"`).


In [0]:
# Cast timestamp to date, cast total_amount to decimal
customers_casted = customers_df.withColumn(
    "reg_date_only", col("registration_date").cast(DateType())
)
display(customers_casted.select("customer_id", "registration_date", "reg_date_only").limit(5))

orders_casted = orders_df.withColumn(
    "total_amount_decimal", col("total_amount").cast("decimal(10,2)")
)
display(orders_casted.select("order_id", "total_amount", "total_amount_decimal").limit(5))


### filter / where — Filtering Rows

> **Definition (Databricks):** `DataFrame.filter(condition)` (alias: `where()`) — returns rows satisfying the condition. The condition can be a `Column` expression or a SQL string. `filter()` and `where()` are identical. Spark uses *predicate pushdown* to push filters as close to the data source as possible.

| Pattern | Example |
|---|---|
| Equality | `filter(col("country") == "USA")` |
| AND | `filter((col("country") == "USA") & (col("city") == "NY"))` |
| OR | `filter((col("country") == "USA") \| (col("country") == "UK"))` |
| Null check | `filter(col("email").isNotNull())` |
| List | `filter(col("country").isin(["USA", "UK", "DE"]))` |


In [0]:
# Simple filter
usa_customers = customers_df.filter(col("country") == "USA")
display(usa_customers.select("customer_id", "first_name", "country").limit(5))

# AND condition
usa_2023 = customers_df.filter(
    (col("country") == "USA") & (year(col("registration_date")) == 2023)
)
display(usa_2023.select("customer_id", "first_name", "country", "registration_date").limit(5))

# SQL-style string condition (equivalent)
recent_orders = orders_df.filter("total_amount > 100")
print(f"Orders > $100: {recent_orders.count()}")


### when / otherwise — Conditional Logic

> **Definition (Databricks):** `when(condition, value)` — the PySpark equivalent of SQL `CASE WHEN ... THEN ... ELSE ... END`. Returns a `Column` expression. Chain multiple `.when()` calls for additional branches. `.otherwise(default)` is optional — without it, unmatched rows return `null`.

| Pattern | SQL equivalent |
|---|---|
| `when(cond, val)` | `CASE WHEN cond THEN val END` |
| `when(c1,v1).when(c2,v2)` | `CASE WHEN c1 THEN v1 WHEN c2 THEN v2 END` |
| `when(c1,v1).otherwise(v2)` | `CASE WHEN c1 THEN v1 ELSE v2 END` |


In [0]:
# Classify customers by registration year
customers_tiered = customers_df.withColumn(
    "customer_tier",
    when(year(col("registration_date")) >= 2023, "New")
    .when(year(col("registration_date")) >= 2021, "Regular")
    .otherwise("Loyal")
)
display(customers_tiered.select("customer_id", "first_name", "registration_date", "customer_tier").limit(10))


### groupBy + agg — Aggregations

> **Definition (Databricks):** `DataFrame.groupBy(*cols)` — groups rows by specified columns, returning a `GroupedData` object. Use aggregate functions (`count()`, `sum()`, `avg()`, `min()`, `max()`) or the generic `agg()` for multiple metrics in one pass. SQL equivalent: `GROUP BY` + aggregate functions.


In [0]:
# Count customers by country
customers_by_country = (customers_df
    .groupBy("country")
    .count()
    .orderBy(desc("count"))
)
display(customers_by_country.limit(10))

# Multiple aggregations: order stats by payment method
order_stats = (orders_df
    .groupBy("payment_method")
    .agg(
        count("*").alias("order_count"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("avg_order_value")
    )
    .orderBy(desc("total_revenue"))
)
display(order_stats)


In [0]:
# Tables created during this demo
created_tables = [
    "customer_cts",
    "customer_ctas_v2",
    "customers",  # created by read_files() CTAS
]

## Cleanup

Remove demo tables, checkpoints, and temporary data created during this module.

In [0]:
results = []
for table in created_tables:
    full_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table}"
    try:
        if spark.catalog.tableExists(full_table):
            count = spark.table(full_table).count()
            results.append((table, "EXISTS", str(count)))
        else:
            results.append((table, "NOT FOUND", "-"))
    except Exception as e:
        results.append((table, "ERROR", str(e)[:30]))

display(spark.createDataFrame(results, ["Table", "Status", "Records"]))

In [0]:
# Cleanup flag
CLEANUP_ENABLED = False

**Execute cleanup (if enabled):**

In [0]:
if CLEANUP_ENABLED:
    results = []
    for table in created_tables:
        full_table = f"{CATALOG}.{BRONZE_SCHEMA}.{table}"
        try:
            spark.sql(f"DROP TABLE IF EXISTS {full_table}")
            results.append((table, "DROPPED"))
        except Exception as e:
            results.append((table, f"ERROR: {str(e)[:30]}"))
    
    # Cleanup checkpoints
    try:
        dbutils.fs.rm(CHECKPOINT_BASE_PATH, True)
        results.append(("checkpoints", "REMOVED"))
    except:
        results.append(("checkpoints", "NOT FOUND"))
    
    display(spark.createDataFrame(results, ["Resource", "Status"]))
else:
    display(spark.createDataFrame([
        ("CLEANUP_ENABLED", "False"),
        ("Action", "Change to True to delete resources")
    ], ["Setting", "Value"]))

← [01 — Platform & Workspace](01_platform_and_workspace.ipynb) | **[ README](../../../README.md)** | [03 — Delta Lake →](03_delta_lake.ipynb)